In [ ]:
# Loading libraries
from deepspot import DeepSpot
from deepspot.utils.utils_image import predict_spatial_transcriptomics_from_image_path
from deepspot.utils.utils_image import get_morphology_model_and_preprocess
from deepspot.utils.utils_image import crop_tile


from huggingface_hub import login, hf_hub_download
import matplotlib.image as mpimg
from openslide import open_slide
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import squidpy as sq
import anndata as ad
import pandas as pd
import numpy as np
import pyvips
import torch
import math
import yaml
import json
import PIL
import timm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
input_image = snakemake.input.input_image
image_config_path = snakemake.input.image_config
read_count_table_path = snakemake.output.read_count_table

In [ ]:
# Loading image configs
with open(image_config_path) as f:
    image_config = json.load(f)
deepspot_pretrained_model_name = image_config["model_name"]
white_cutoff = image_config["white_cutoff"]
downsample_factor = image_config["downsample_factor"]

In [ ]:
# Loading pre-trained deepspot model weights, configs and foundation model
deepspot_model_path = getattr(snakemake.input, deepspot_pretrained_model_name + "_pretrained")

deepspot_model_directory = Path(deepspot_model_path).parent
tissue_hvg = deepspot_model_directory / "info_highly_variable_genes.csv"
model_weights = deepspot_model_directory / "final_model.pkl"
model_params = deepspot_model_directory / "top_param_overall.yaml"

In [ ]:
# Loading model_params, foundation_model_params
with open(model_params, "r") as stream:
    config = yaml.safe_load(stream)

foundation_model_name = config["image_feature_model"]
foundation_model_weights = getattr(snakemake.input, foundation_model_name)

In [ ]:
# Load grid parameters from model_hparams
n_mini_tiles = config['n_mini_tiles'] # number of non-overlaping subspots
spot_diameter = config['spot_diameter'] # spot diameter
spot_distance = config['spot_distance'] # distance between spots
sample = 'image'

In [ ]:
genes = pd.read_csv(tissue_hvg)
selected_genes_bool = genes.isPredicted.values
genes_to_predict = genes[selected_genes_bool]
genes_to_predict.sort_values("highly_variable_rank")

In [ ]:
# Load the image
image = pyvips.Image.new_from_file(input_image)

# Display the image
plt.imshow(image)
plt.axis('off')  # Turn off axis labels
plt.show()

In [ ]:
coord = []
for i, x in enumerate(range(spot_diameter + 1, image.height - spot_diameter - 1, spot_distance)):
    for j, y in enumerate(range(spot_diameter + 1, image.width - spot_diameter - 1, spot_distance)):
        coord.append([i, j, x, y])
coord = pd.DataFrame(coord, columns=['x_array', 'y_array', 'x_pixel', 'y_pixel'])
coord.index = coord.index.astype(str)

In [ ]:
is_white = []
counts = []
for _, row in tqdm(coord.iterrows()):
    x = row.x_pixel - int(spot_diameter // 2)
    y = row.y_pixel - int(spot_diameter // 2)
    
    main_tile = crop_tile(image, x, y, spot_diameter)
    main_tile = main_tile[:,:,:3]
    white = np.mean(main_tile)
    is_white.append(white)

counts = np.empty((len(is_white), selected_genes_bool.sum())) # empty count matrix 

coord['is_white'] = is_white

In [ ]:
adata = ad.AnnData(counts)
adata.var.index = genes[selected_genes_bool].gene_name.values
adata.obs = adata.obs.merge(coord, left_index=True, right_index=True)
adata.obs['is_white'] = coord['is_white'].values
adata.obs['is_white_bool'] = (coord['is_white'].values > white_cutoff).astype(int)
adata.obs['sampleID'] = sample
adata.obs['barcode'] = adata.obs.index
adata = adata[adata.obs.is_white_bool == 0, ]
adata

In [ ]:
### CREATE IMAGE
img = open_slide(input_image)
n_level = len(img.level_dimensions) - 1 # 0 based


large_w, large_h = img.dimensions
new_w = math.floor(large_w / downsample_factor)
new_h = math.floor(large_h / downsample_factor)

whole_slide_image = img.read_region((0, 0), n_level, img.level_dimensions[-1])
whole_slide_image = whole_slide_image.convert("RGB")
img_downsample = whole_slide_image.resize((new_w, new_h), PIL.Image.BILINEAR)


adata.obsm['spatial'] = adata.obs[["y_pixel", "x_pixel"]].values
# adjust coordinates to new image dimensions
adata.obsm['spatial'] = adata.obsm['spatial'] / downsample_factor
# create 'spatial' entries
adata.uns['spatial'] = dict()
adata.uns['spatial']['library_id'] = dict()
adata.uns['spatial']['library_id']['images'] = dict()
adata.uns['spatial']['library_id']['images']['hires'] = np.array(img_downsample)

In [ ]:
model_hparam = config

In [ ]:
model_expression = torch.load(model_weights, map_location=device)
model_expression.to(device)
model_expression.eval()

In [ ]:
morphology_model, preprocess, feature_dim = get_morphology_model_and_preprocess(model_name=foundation_model_name, 
                                                                                device=device, model_path=foundation_model_weights)
morphology_model.to(device)
morphology_model.eval()

In [ ]:
counts = predict_spatial_transcriptomics_from_image_path(input_image, 
                                                        adata,
                                                        spot_diameter,
                                                        n_mini_tiles,
                                                        preprocess, 
                                                        morphology_model, 
                                                        model_expression, 
                                                        device,
                                                        super_resolution=False,
                                                        neighbor_radius=1)

In [ ]:
counts.shape

In [ ]:
counts_norm = model_expression.inverse_transform(counts)
counts_norm[counts_norm < 0] = 0

In [ ]:
adata_predicted = ad.AnnData(counts_norm, 
                             var=adata.var,
                             obs=adata.obs, 
                             uns=adata.uns, 
                             obsm=adata.obsm).copy()
adata_predicted

In [ ]:
sq.pl.spatial_scatter(adata_predicted, 
                      color=['MUC2'], 
                      wspace=0,
                      ncols=4,
                      size=6)

In [ ]:
# Adjusting to cellxgene browser requirements
# NOTE img_downsampled needs to be adjusted based on the size of the input dataset by the downsample_factor hyperparameter
adata_predicted.uns['20x_slide'] = np.array(img_downsample)
adata_predicted.uns["image_extents"] = np.array([new_w, new_h])
adata_predicted.obsm["X_spatial"] = pd.concat((adata.obs['x_pixel'] / downsample_factor,
                                               adata.obs['y_pixel'] / downsample_factor), 
                                              axis=1).to_numpy()

In [ ]:
adata_predicted.X.astype(np.float64).max()

In [ ]:
np.expm1(adata_predicted.X).max()

In [ ]:
# adjusting for cellwhisperer
# 1. Enusre raw counts are stored in int32
adata_predicted.layers["counts"] = np.expm1(adata_predicted.X.astype(np.float64)).astype(np.int32)

In [ ]:
# 2 Ensure var index is unique and add gene_name
assert adata_predicted.var.index.is_unique, "Error: var index is not unique!"
adata_predicted.var['gene_name'] = adata_predicted.var.index.values

In [ ]:
# 3 add default_embedding
adata_predicted.uns['default_embedding'] = "X_spatial"

In [ ]:
output_path = snakemake.output.read_count_table
adata_predicted.write(output_path)
print(adata_predicted)
print(f"Saved CellWhisperer-compatible file to {output_path}")